# PathSnap - P2: Model Experimentation + Bayesian Optimisation
Expects P1's candidate tables (parquet): columns `trip_id, obs_idx, arc_id, label` + numeric feature columns.
`label`=1 if the candidate arc is the reference arc for that GPS observation.
Train = 80 trips, student-test = 10 trips. **Never load evaluator trips here.**

In [ ]:
!pip -q install optuna lightgbm pyarrow

In [ ]:
import time, json, os, warnings
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, log_loss, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier
import optuna
warnings.filterwarnings('ignore')

SEED = 42
TRAIN_PATH   = 'candidates_train.parquet'     # <- from P1 (80 trips)
STUDENT_PATH = 'candidates_student.parquet'   # <- from P1 (10 student-test trips)
META = ['trip_id', 'obs_idx', 'arc_id', 'label']
N_FOLDS = 5
os.makedirs('results', exist_ok=True); os.makedirs('models', exist_ok=True)

In [ ]:
# Colab: upload the two parquet files (or mount Drive and edit the paths above)
from google.colab import files
if not os.path.exists(TRAIN_PATH):
    files.upload()

In [ ]:
tr = pd.read_parquet(TRAIN_PATH).reset_index(drop=True)
st = pd.read_parquet(STUDENT_PATH).reset_index(drop=True)
FEATURES = [c for c in tr.columns if c not in META and pd.api.types.is_numeric_dtype(tr[c])]
print(len(FEATURES), 'features:', FEATURES)
print('train', tr.shape, tr.trip_id.nunique(), 'trips | student', st.shape, st.trip_id.nunique(), 'trips')
assert set(tr.trip_id).isdisjoint(set(st.trip_id)), 'trip leakage between partitions!'
print('pos rate', tr.label.mean().round(4))
def miss_rate(d): return 1 - d.groupby(['trip_id','obs_idx']).label.max().mean()
print('candidate miss rate  train %.4f | student %.4f' % (miss_rate(tr), miss_rate(st)))

## Metrics
Top-1 is observation-level: per GPS point, take the highest-scoring candidate. Observations whose reference arc is not in the candidate set count as wrong. Macro-F1 / log loss are candidate-level binary (threshold 0.5) - **confirm with the team this matches the report's definition.**

In [ ]:
def pick_top1(df, p):
    d = df[['trip_id','obs_idx','arc_id','label']].copy(); d['p'] = p
    return d.loc[d.groupby(['trip_id','obs_idx'])['p'].idxmax()]

def build_adj(arcs):  # arcs: (n,2) int array [from_node,to_node]; arc_id = row index
    from collections import defaultdict
    by_from = defaultdict(list)
    for i,(a,b) in enumerate(arcs): by_from[a].append(i)
    return {i: set(by_from[b]) for i,(a,b) in enumerate(arcs)}

ADJ = None  # dict trip_id -> build_adj(arcs of that trip). Ask P1/P3 to fill; None => continuity N/A

def continuity(picks, adj=ADJ):
    if adj is None: return np.nan
    ok = tot = 0
    for t, g in picks.sort_values(['trip_id','obs_idx']).groupby('trip_id'):
        a, nxt = g.arc_id.values, adj[t]
        for x, y in zip(a[:-1], a[1:]):
            tot += 1; ok += (x == y) or (y in nxt.get(x, ()))
    return ok / tot if tot else np.nan

def evaluate(model, df):
    X = df[FEATURES].values
    t = time.perf_counter(); p = model.predict_proba(X)[:,1]; inf = time.perf_counter() - t
    picks = pick_top1(df, p)
    n_obs = df.groupby(['trip_id','obs_idx']).ngroups
    return dict(top1=picks.label.sum()/n_obs,
                macro_f1=f1_score(df.label, p >= .5, average='macro'),
                logloss=log_loss(df.label, np.clip(p,1e-7,1-1e-7), labels=[0,1]),
                continuity=continuity(picks),
                inf_ms_per_1k=inf/len(df)*1e6), p

## 5 models, 5 families

In [ ]:
MODELS = {
 'LogReg (linear)': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
 'KNN (instance)':  make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1)),
 'RandomForest (bagging)': RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
 'LightGBM (boosting)': LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, random_state=SEED, verbose=-1),
 'MLP (neural)': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-4, early_stopping=True, max_iter=200, random_state=SEED)),
}
KEY_CFG = {  # for the report's 'Key Configuration' column
 'LogReg (linear)': 'C=1, lbfgs, scaled', 'KNN (instance)': 'k=15, distance-weighted, scaled',
 'RandomForest (bagging)': '200 trees, min_leaf=2', 'LightGBM (boosting)': '300 est, lr=0.1, leaves=31',
 'MLP (neural)': '(64,32), adam, early stop, scaled'}

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
Xtr, ytr, gtr = tr[FEATURES].values, tr.label.values, tr.trip_id.values
rows, fitted, student_p = [], {}, {}
for name, m in MODELS.items():
    cvf1, cvt1 = [], []
    for fi, (a, b) in enumerate(gkf.split(Xtr, ytr, gtr)):
        mm = clone(m).fit(Xtr[a], ytr[a])
        r, _ = evaluate(mm, tr.iloc[b]); cvf1.append(r['macro_f1']); cvt1.append(r['top1'])
    t0 = time.perf_counter(); full = clone(m).fit(Xtr, ytr); fit_s = time.perf_counter() - t0
    rtr, _ = evaluate(full, tr); rst, p = evaluate(full, st)
    fitted[name], student_p[name] = full, p
    rows.append(dict(model=name, config=KEY_CFG[name], cv_f1_mean=np.mean(cvf1), cv_f1_std=np.std(cvf1),
                     cv_top1=np.mean(cvt1), train_top1=rtr['top1'], train_s=fit_s,
                     st_top1=rst['top1'], st_macro_f1=rst['macro_f1'], st_logloss=rst['logloss'],
                     st_continuity=rst['continuity'], st_inf_ms_per_1k=rst['inf_ms_per_1k']))
    print(name, {k: round(v,4) for k,v in rows[-1].items() if isinstance(v,float)})
    joblib.dump(full, f"models/{name.split()[0]}.joblib")
comp = pd.DataFrame(rows).round(4); comp.to_csv('results/model_comparison.csv', index=False)
comp

Overfitting check: compare `train_top1` vs `cv_top1` vs `st_top1` in the table above (big train-to-CV gap = overfit).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16,4)); n = comp.model.str.split().str[0]
ax[0].bar(n, comp.cv_f1_mean, yerr=comp.cv_f1_std); ax[0].set_title('CV Macro F1 (mean±std, 5-fold group)')
w = .38; x = np.arange(len(n))
ax[1].bar(x-w/2, comp.st_top1, w, label='Top-1'); ax[1].bar(x+w/2, comp.st_macro_f1, w, label='Macro F1')
ax[1].set_xticks(x); ax[1].set_xticklabels(n); ax[1].legend(); ax[1].set_title('Student-test')
ax[2].bar(n, comp.st_logloss); ax[2].set_title('Student-test Log Loss')
for a in ax: a.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('results/model_comparison.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 5, figsize=(20,3.6))
for a,(k,p) in zip(ax, student_p.items()):
    ConfusionMatrixDisplay.from_predictions(st.label, p>=.5, ax=a, colorbar=False); a.set_title(k.split()[0])
plt.tight_layout(); plt.savefig('results/confusion_matrices.png', dpi=150); plt.show()

# only models with real epoch histories (MLP) get a loss curve
mlp = fitted['MLP (neural)'][-1]
plt.plot(mlp.loss_curve_, label='train loss')
if getattr(mlp, 'validation_scores_', None) is not None: plt.plot(mlp.validation_scores_, label='val score (accuracy)')
plt.legend(); plt.xlabel('epoch'); plt.title('MLP training history'); plt.savefig('results/mlp_curve.png', dpi=150); plt.show()

## Bayesian hyperparameter optimisation (Optuna TPE, group-aware CV)
Pick `BEST` from the table (primary metric: CV Macro F1, tie-break on student top-1 / log loss / inference time). Search space below is for LightGBM - edit if you pick another model.

In [ ]:
BEST = 'LightGBM (boosting)'
N_TRIALS = 40

def make_lgbm(**kw): return LGBMClassifier(random_state=SEED, verbose=-1, subsample_freq=1, **kw)

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True))
    f1s = []
    for a, b in gkf.split(Xtr, ytr, gtr):
        m = make_lgbm(**params).fit(Xtr[a], ytr[a])
        r, _ = evaluate(m, tr.iloc[b]); f1s.append(r['macro_f1'])
    return float(np.mean(f1s))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
t0 = time.perf_counter(); study.optimize(objective, n_trials=N_TRIALS); opt_s = time.perf_counter() - t0
study.trials_dataframe().to_csv('results/optuna_trials.csv', index=False)
json.dump(study.best_params, open('results/best_params.json','w'), indent=2)
print('best CV macro F1 %.4f in %.0fs' % (study.best_value, opt_s)); print(study.best_params)

In [ ]:
tuned = make_lgbm(**study.best_params).fit(Xtr, ytr)
joblib.dump(tuned, 'models/final_tuned.joblib')
untuned = fitted[BEST]
def cv_f1(m):
    return np.mean([evaluate(clone(m).fit(Xtr[a], ytr[a]), tr.iloc[b])[0]['macro_f1'] for a,b in gkf.split(Xtr,ytr,gtr)])
out = []
for nm, m in [('untuned', untuned), ('tuned', tuned)]:
    r, _ = evaluate(m, st); r['cv_macro_f1'] = cv_f1(m); out.append(dict(model=nm, **r))
tt = pd.DataFrame(out).round(4); tt.to_csv('results/tuned_vs_untuned.csv', index=False); tt

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
plot_optimization_history(study); plt.savefig('results/optuna_history.png', dpi=150, bbox_inches='tight'); plt.show()
plot_param_importances(study);    plt.savefig('results/optuna_importances.png', dpi=150, bbox_inches='tight'); plt.show()
print('Hand off to P3/P4: models/final_tuned.joblib + FEATURES order ->'); print(FEATURES)
json.dump(FEATURES, open('models/features.json','w'))

In [ ]:
!zip -qr p2_outputs.zip results models
files.download('p2_outputs.zip')